# 1. Parsowanie Plików

## Modele Danych

In [1]:
from BayesianNetwork.models.data import Data
from BayesianNetwork.models.graph import GraphData
from BayesianNetwork.models.variable import Variable
from BayesianNetwork.models.queries import Query, QueryType

## Wybór katalogu danych


In [2]:
import os
REPO = "k12"

data_path = os.path.join('..', 'data', 'zestawy', REPO)
files = os.listdir(data_path)

files

['zapytania.txt', 'dane.csv', 'test.csv', 'ograniczenia.txt', 'opis.json']

## Parsowanie do odpowiednich Modeli

In [4]:
from BayesianNetwork.parsing.description_parsing import description_parser
from rich import print as rprint

desc_path = os.path.join(data_path, 'opis.json')
graf = description_parser(desc_path)

rprint(graf)

GraphData(
    variables_dict={
        'D': Variable(name='D', description='trudność kursu', states=['Easy', 'Hard']),
        'I': Variable(name='I', description='zdolności studenta', states=['Dumb', 'Intelligent']),
        'M': Variable(name='M', description='motywacja studenta', states=['Low', 'High']),
        'W': Variable(name='W', description='praca zarobkowa w trakcie studiów', states=['No', 'Yes']),
        'A': Variable(name='A', description='frekwencja na zajęciach', states=['Low', 'Medium', 'High']),
        'G': Variable(name='G', description='ocena z kursu', states=['A', 'B', 'C']),
        'S': Variable(name='S', description='wynik testu SAT', states=['Bad', 'Good']),
        'L': Variable(name='L', description='list polecający', states=['Bad', 'Good']),
        'B': Variable(name='B', description='stypendium naukowe', states=['No', 'Yes'])
    },
    exper_edges=[
        ('A', 'L'),
        ('D', 'A'),
        ('D', 'G'),
        ('G', 'B'),
        ('G', 'L'),
        ('I', 'D'),
        ('I', 'G'),
        ('I', 'L'),
        ('I', 'S'),
        ('M', 'G'),
        ('M', 'S'),
        ('W', 'A')
    ],
    required_edges=[('D', 'G'), ('G', 'L'), ('I', 'G'), ('I', 'S')],
    forbidden_edges=[('A', 'G'), ('B', 'L'), ('D', 'B'), ('M', 'B')],
    tiers=[['D', 'I', 'M', 'W'], ['A'], ['G', 'S'], ['L', 'B']]
)

In [9]:
from BayesianNetwork.parsing.data_parsing import parse_data

variables = graf.variables_dict
train_data_path = os.path.join(data_path, 'dane.csv')
train =  parse_data(
    path = train_data_path,
    variables = variables,
    is_train_data=True,
    allow_missing=True
)

print(train.data_purpose)
train.data_frame

DataPurpose.Train


/tmp/ipykernel_29380/1303061874.py:5: UserWarning: Train data contains 4740 nans
  train =  parse_data(


,D,I,M,W,A,G,S,L,B
0,Easy,NaN,NaN,No,High,B,Bad,Bad,No
1,Easy,Dumb,NaN,No,Medium,A,Bad,NaN,Yes
2,NaN,Dumb,Low,NaN,Low,C,Bad,Bad,No
3,Easy,NaN,High,No,NaN,NaN,Bad,Good,Yes
4,Hard,NaN,High,No,Medium,C,Bad,Bad,No
...,...,...,...,...,...,...,...,...,...
2995,Easy,NaN,Low,Yes,Medium,C,Bad,Bad,No
2996,Easy,NaN,High,No,High,A,Bad,NaN,Yes
2997,Easy,NaN,High,NaN,Medium,A,NaN,Good,Yes
2998,Hard,NaN,High,No,Low,C,NaN,Bad,No


In [7]:
test_data_path = os.path.join(data_path, 'test.csv')
test = parse_data(
    path = test_data_path,
    variables = variables,
    is_train_data=False,
    allow_missing=False
)

print(test.data_purpose)
test.data_frame

DataPurpose.Test


,D,I,M,W,A,G,S,L,B
0,Easy,Dumb,High,Yes,High,A,Bad,Good,Yes
1,Hard,Dumb,Low,Yes,Low,C,Bad,Bad,No
2,Hard,Dumb,Low,No,Low,C,Bad,Bad,No
3,Easy,Dumb,High,No,High,A,Bad,Good,Yes
4,Hard,Dumb,Low,No,Low,C,Good,Bad,No
...,...,...,...,...,...,...,...,...,...
1995,Hard,Dumb,Low,Yes,Medium,C,Bad,Bad,No
1996,Hard,Intelligent,Low,No,Low,A,Bad,Good,Yes
1997,Hard,Dumb,High,No,Low,C,Bad,Bad,No
1998,Easy,Dumb,High,No,High,A,Bad,Good,Yes


In [10]:
from BayesianNetwork.parsing.query_parsing import query_parser
queries = query_parser(
    path=os.path.join(data_path, 'zapytania.txt'),
    variables_dict=variables,
)

queries

[Query(query_type=<QueryType.P: 1>, variables=[Variable(name='G', description='ocena z kursu', states=['A', 'B', 'C'])], evidence={}),
 Query(query_type=<QueryType.P: 1>, variables=[Variable(name='L', description='list polecający', states=['Bad', 'Good'])], evidence={}),
 Query(query_type=<QueryType.P: 1>, variables=[Variable(name='I', description='zdolności studenta', states=['Dumb', 'Intelligent'])], evidence={}),
 Query(query_type=<QueryType.P: 1>, variables=[Variable(name='B', description='stypendium naukowe', states=['No', 'Yes'])], evidence={}),
 Query(query_type=<QueryType.P: 1>, variables=[Variable(name='G', description='ocena z kursu', states=['A', 'B', 'C'])], evidence={'D': 'Hard', 'I': 'Intelligent'}),
 Query(query_type=<QueryType.P: 1>, variables=[Variable(name='G', description='ocena z kursu', states=['A', 'B', 'C'])], evidence={'D': 'Easy', 'I': 'Dumb'}),
 Query(query_type=<QueryType.P: 1>, variables=[Variable(name='I', description='zdolności studenta', states=['Dumb', '

# 2. Sieć z Grafu Eksperckiego